In [1]:
import os
import json
import time
import itertools
import random

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from IPython.display import display

In [5]:
# ============================================================
# HYPERPARAMETER SEARCH SPACE
# ============================================================

WEIGHT_DECAYS = [0, 1e-4, 1e-3, 1e-1, 0.2]
HIDDEN_WIDTHS = [20, 48, 128, 256, 512]
LABEL_NOISES = [0, 5, 10, 15, 20, 25]  # Percentages


# ============================================================
# TRAINING PARAMETERS
# ============================================================

EPOCHS = 10_000
BATCH_SIZE = 512
LEARNING_RATE = 1e-3

OPTIMIZER = "AdamW"

# Dataset
MODULUS = 97
TRAIN_FRACTION = 0.70

# Random seeds
DATA_SEED = 42
MODEL_SEED = 42


# ============================================================
# EVALUATION & CHECKPOINTS
# ============================================================

EVAL_EVERY = 50
CHECKPOINT_EVERY = 500

TRAIN_ACCURACY_THRESHOLD = 0.99
GROKKING_MIN_DELAY = 200

# Save final trained models
SAVE_MODELS = True

# None = all models
# Example: [0, 1, 2] = only train those indices
RUN_INDICES = None


# ============================================================
# DIRECTORIES
# ============================================================

OUT_DIR = Path("grokking_experiments")
MODELS_DIR = OUT_DIR / "models"

DATASET_DIR = OUT_DIR / "datasets"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_FILE = OUT_DIR / "configurations.json"


# ============================================================
# HARDWARE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {DEVICE}")
print(f"Epochs per model: {EPOCHS:,}")
print(f"Checkpoints every: {CHECKPOINT_EVERY} epochs")

Device: cuda
Epochs per model: 10,000
Checkpoints every: 500 epochs


In [3]:
# ============================================================
# GENERATE ALL PARAMETER COMBINATIONS
# ============================================================

configurations = []

for index, (wd, width, noise) in enumerate(
    itertools.product(
        WEIGHT_DECAYS,
        HIDDEN_WIDTHS,
        LABEL_NOISES
    )
):
    configurations.append({
        "index": index,
        "weightDecay": float(wd),
        "hiddenWidth": int(width),
        "labelNoise": int(noise)
    })


# ============================================================
# SAVE CONFIGURATIONS TO JSON
# ============================================================

with open(CONFIG_FILE, "w") as f:
    json.dump(configurations, f, indent=4)


# ============================================================
# DISPLAY CONFIGURATIONS
# ============================================================

config_df = pd.DataFrame(configurations)

print(f"Total configurations: {len(configurations)}")
print(f"Total training epochs: {len(configurations) * EPOCHS:,}")
print(f"Configuration file: {CONFIG_FILE}")

display(config_df)

Total configurations: 150
Total training epochs: 1,500,000
Configuration file: grokking_experiments/configurations.json


,index,weightDecay,hiddenWidth,labelNoise
0,0,0.0,20,0
1,1,0.0,20,5
2,2,0.0,20,10
3,3,0.0,20,15
4,4,0.0,20,20
...,...,...,...,...
145,145,0.2,512,5
146,146,0.2,512,10
147,147,0.2,512,15
148,148,0.2,512,20


In [6]:
# ============================================================
# DATASET GENERATION
# ============================================================

# Generate all possible input pairs
a, b = np.meshgrid(
    np.arange(MODULUS),
    np.arange(MODULUS),
    indexing="ij"
)

X = np.column_stack((a.ravel(), b.ravel()))

# Correct labels: (a + b) mod 97
y = (X[:, 0] + X[:, 1]) % MODULUS


# ============================================================
# FIXED TRAIN / TEST SPLIT
# ============================================================

rng = np.random.default_rng(DATA_SEED)

indices = rng.permutation(len(X))
train_size = int(len(X) * TRAIN_FRACTION)

train_indices = indices[:train_size]
test_indices = indices[train_size:]

X_train = X[train_indices]
y_train_clean = y[train_indices]

X_test = X[test_indices]
y_test = y[test_indices]


# ============================================================
# GENERATE AND SAVE SIX DATASETS
# ============================================================

noise_rng = np.random.default_rng(DATA_SEED + 1)

# Same corruption order across noise levels
noise_order = noise_rng.permutation(train_size)

# Nonzero offsets guarantee incorrect labels
noise_offsets = noise_rng.integers(
    1, MODULUS, size=train_size
)

for noise in LABEL_NOISES:

    y_train = y_train_clean.copy()

    n_corrupt = round(train_size * noise / 100)

    corrupt_indices = noise_order[:n_corrupt]

    y_train[corrupt_indices] = (
        y_train[corrupt_indices]
        + noise_offsets[corrupt_indices]
    ) % MODULUS

    # Combine training and testing data
    dataset_df = pd.DataFrame({
        "a": np.concatenate((X_train[:, 0], X_test[:, 0])),
        "b": np.concatenate((X_train[:, 1], X_test[:, 1])),
        "label": np.concatenate((y_train, y_test)),
        "cleanLabel": np.concatenate((y_train_clean, y_test)),
        "split": ["train"] * train_size + ["test"] * len(X_test)
    })

    # Save dataset
    dataset_path = DATASET_DIR / f"dataset_{noise}.csv"

    dataset_df.to_csv(dataset_path, index=False)

    print(
        f"Saved: {dataset_path.name} | "
        f"Noise: {noise}% | "
        f"Corrupted: {n_corrupt}/{train_size}"
    )

print(f"\nAll {len(LABEL_NOISES)} datasets created successfully!")

Saved: dataset_0.csv | Noise: 0% | Corrupted: 0/6586
Saved: dataset_5.csv | Noise: 5% | Corrupted: 329/6586
Saved: dataset_10.csv | Noise: 10% | Corrupted: 659/6586
Saved: dataset_15.csv | Noise: 15% | Corrupted: 988/6586
Saved: dataset_20.csv | Noise: 20% | Corrupted: 1317/6586
Saved: dataset_25.csv | Noise: 25% | Corrupted: 1646/6586

All 6 datasets created successfully!


In [ ]:
# ============================================================
# DATASET VALIDATION
# ============================================================

reference = pd.read_csv(DATASET_DIR / "dataset_0.csv")

expected_train = int(MODULUS**2 * TRAIN_FRACTION)
expected_test = MODULUS**2 - expected_train

previous_corrupted = set()

for noise in sorted(LABEL_NOISES):

    dataset = pd.read_csv(
        DATASET_DIR / f"dataset_{noise}.csv"
    )

    # Verify dataset size
    assert len(dataset) == MODULUS**2

    # Verify identical inputs and splits
    assert dataset[["a", "b", "split"]].equals(
        reference[["a", "b", "split"]]
    )

    # Verify unique input pairs
    assert not dataset.duplicated(["a", "b"]).any()

    # Verify correct mathematical labels
    assert (
        dataset["cleanLabel"]
        == (dataset["a"] + dataset["b"]) % MODULUS
    ).all()

    train = dataset[dataset["split"] == "train"]
    test = dataset[dataset["split"] == "test"]

    assert len(train) == expected_train
    assert len(test) == expected_test

    # Test labels must always be clean
    assert (
        test["label"] == test["cleanLabel"]
    ).all()

    # Count corrupted training labels
    corrupted = set(
        train.index[
            train["label"] != train["cleanLabel"]
        ]
    )

    expected_corrupted = round(
        expected_train * noise / 100
    )

    assert len(corrupted) == expected_corrupted

    # Previous corruptions must remain corrupted
    assert previous_corrupted.issubset(corrupted)

    previous_corrupted = corrupted

    print(
        f"Dataset {noise:2d}% | "
        f"Train: {len(train)} | "
        f"Test: {len(test)} | "
        f"Corrupted: {len(corrupted)} | "
        f"PASSED"
    )

print("\nAll dataset validation tests passed!")